In [1]:
import pandas as pd
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive
from src.analysis.poissonFunctions import (
    compute_bayesian_lambda,
    compute_bayesian_lambda_assists,
    compute_bayesian_lambda_rebounds,
    compute_bayesian_lambda_blocks,
    compute_bayesian_lambda_steals
)

from scipy.stats import poisson
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
s26.rename(columns={'BLK_x': 'BLK'}, inplace=True)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_DFS_20251206_134221.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Michael Porter Jr,Over,26.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
1,PrizePicks,player_points,Michael Porter Jr,Under,26.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
2,PrizePicks,player_points,Trey Murphy III,Over,20.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
3,PrizePicks,player_points,Trey Murphy III,Under,20.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
4,PrizePicks,player_points,Jeremiah Fears,Over,16.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21


## Points

### prizepicks

In [3]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/prizepicks/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,P.J. Washington,13.5,0.8,0.7,0.67,0.840,0.160
1,Rudy Gobert,9.5,0.6,0.6,0.60,0.815,0.185
2,Micah Peavy,4.5,0.8,0.7,0.47,0.804,0.196
3,Jaylon Tyson,12.5,0.8,0.7,0.67,0.796,0.204
4,Anthony Edwards,29.5,0.8,0.6,0.60,0.779,0.221
5,Noah Clowney,15.5,0.8,0.7,0.67,0.775,0.225
6,Jose Alvarado,9.0,0.8,0.5,0.40,0.742,0.258
7,Kawhi Leonard,23.5,0.8,0.6,0.53,0.731,0.269
8,Donovan Mitchell,28.5,0.6,0.6,0.60,0.721,0.279
9,Naji Marshall,10.5,0.8,0.6,0.60,0.720,0.280


### underdog

In [4]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/underdog/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,P.J. Washington,13.5,0.8,0.7,0.67,0.840,0.160
1,Rudy Gobert,9.5,0.6,0.6,0.60,0.815,0.185
2,Micah Peavy,4.5,0.8,0.7,0.47,0.804,0.196
3,Jaylon Tyson,12.5,0.8,0.7,0.67,0.796,0.204
4,Naji Marshall,10.5,0.8,0.6,0.60,0.720,0.280
5,Noah Clowney,16.5,0.6,0.6,0.60,0.695,0.305
6,Duncan Robinson,11.5,0.4,0.7,0.60,0.694,0.306
7,Bryce McGowens,8.5,0.4,0.4,0.27,0.682,0.318
8,Ryan Nembhard,9.5,0.8,0.4,0.27,0.672,0.328
9,Nickeil Alexander-Walker,20.5,1.0,0.8,0.60,0.668,0.332


## Assists

### prizepicks

In [5]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/prizepicks/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Rudy Gobert,1.5,1.0,0.8,0.60,0.695,0.305
1,Jalen Johnson,8.0,0.4,0.5,0.40,0.570,0.430
2,Russell Westbrook,7.0,0.4,0.4,0.47,0.554,0.446
3,Darius Garland,6.5,0.8,0.5,0.33,0.535,0.465
4,Donte DiVincenzo,3.5,0.6,0.4,0.47,0.505,0.495
5,Davion Mitchell,8.0,0.8,0.5,0.40,0.496,0.504
6,Saddiq Bey,2.5,0.6,0.5,0.53,0.487,0.513
7,Kyshawn George,4.5,0.6,0.7,0.53,0.483,0.517
8,Mike Conley,2.5,0.4,0.5,0.53,0.480,0.520
9,Julius Randle,5.5,0.6,0.5,0.47,0.477,0.523


### underdog

In [6]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/underdog/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Rudy Gobert,1.5,1.0,0.8,0.60,0.695,0.305
1,Naz Reid,2.5,0.8,0.5,0.53,0.541,0.459
2,Darius Garland,6.5,0.8,0.5,0.33,0.535,0.465
3,Keegan Murray,1.5,0.6,0.3,0.20,0.524,0.476
4,Mike Conley,2.5,0.4,0.5,0.53,0.480,0.520
5,Julius Randle,5.5,0.6,0.5,0.47,0.477,0.523
6,Kris Dunn,2.5,0.4,0.6,0.60,0.471,0.529
7,Anthony Edwards,4.5,0.6,0.4,0.47,0.455,0.545
8,Tyrese Martin,2.5,0.2,0.5,0.40,0.417,0.583
9,Kevin Porter Jr.,6.5,0.4,0.2,0.13,0.356,0.644


# REBOUNDS

### prizepicks

In [7]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/prizepicks/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Cade Cunningham,6.0,0.6,0.7,0.47,0.724,0.276
1,P.J. Washington,6.0,1.0,0.9,0.80,0.653,0.347
2,Jaden McDaniels,4.5,0.4,0.5,0.53,0.646,0.354
3,Kel'el Ware,10.0,0.0,0.5,0.53,0.612,0.388
4,Rudy Gobert,10.0,0.6,0.6,0.53,0.591,0.409
5,Darius Garland,2.5,0.4,0.3,0.20,0.555,0.445
6,Dyson Daniels,6.0,0.4,0.6,0.53,0.548,0.452
7,Kevin Durant,5.0,0.6,0.3,0.27,0.545,0.455
8,Bam Adebayo,10.0,0.6,0.4,0.33,0.544,0.456
9,Julius Randle,7.0,0.6,0.5,0.53,0.518,0.482


### underdog

In [8]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/underdog/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Duncan Robinson,2.5,0.6,0.6,0.53,0.712,0.288
1,Yves Missi,5.5,0.8,0.6,0.60,0.646,0.354
2,Nickeil Alexander-Walker,3.5,0.6,0.6,0.47,0.602,0.398
3,Jose Alvarado,2.5,0.8,0.7,0.53,0.597,0.403
4,Darius Garland,2.5,0.4,0.3,0.20,0.555,0.445
5,Kris Dunn,2.5,0.8,0.6,0.47,0.553,0.447
6,Ziaire Williams,2.5,0.6,0.6,0.53,0.516,0.484
7,Kyshawn George,5.5,0.4,0.5,0.40,0.493,0.507
8,Klay Thompson,2.5,0.8,0.4,0.40,0.480,0.520
9,Precious Achiuwa,5.5,0.6,0.4,0.33,0.464,0.536


## Blocks

### prizepicks

In [9]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/prizepicks/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Nickeil Alexander-Walker,0.5,0.4,0.4,0.47,0.459,0.541,2.176
1,Evan Mobley,1.5,0.8,0.6,0.47,0.564,0.436,1.772
2,Ausar Thompson,0.5,0.2,0.4,0.53,0.450,0.550,2.224
3,Jericho Sims,0.5,0.2,0.4,0.27,0.324,0.676,3.084
4,John Collins,0.5,0.6,0.5,0.47,0.539,0.461,1.854
5,Precious Achiuwa,0.5,0.4,0.3,0.33,0.323,0.677,3.097
6,Anthony Davis,1.5,0.6,0.5,0.33,0.500,0.500,2.000


### underdog

In [10]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/underdog/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Evan Mobley,1.5,0.8,0.6,0.47,0.564,0.436,1.772


# STEALS

### prizepicks

In [11]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/prizepicks/player_steals.csv', index=False)
    steals_df.head(10)

### underdog

In [12]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/underdog/player_steals.csv', index=False)
    steals_df.head(10)

### prizepicks

In [13]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/prizepicks/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 78 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Saddiq Bey,25.5,1.0,0.7,0.60
1,Kawhi Leonard,31.5,1.0,0.7,0.60
2,Naji Marshall,17.5,1.0,0.8,0.80
3,Nickeil Alexander-Walker,28.5,1.0,0.8,0.60
4,Micah Peavy,7.5,0.8,0.8,0.60
...,...,...,...,...,...
73,Kyshawn George,25.5,0.2,0.4,0.27
74,Terance Mann,15.0,0.2,0.4,0.47
75,Josh Okogie,11.5,0.2,0.3,0.40
76,Jalen Duren,32.5,0.0,0.3,0.47



Processing player_points_rebounds...
Saved 76 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Naji Marshall,15.5,1.0,0.7,0.73
1,Nickeil Alexander-Walker,24.5,1.0,0.8,0.60
2,Kawhi Leonard,28.5,0.8,0.6,0.53
3,Norman Powell,28.0,0.8,0.7,0.60
4,Jeremiah Fears,20.5,0.8,0.6,0.60
...,...,...,...,...,...
71,Jaden Ivey,11.5,0.2,0.2,0.13
72,Myles Turner,18.5,0.2,0.5,0.60
73,Cade Cunningham,33.5,0.2,0.5,0.53
74,Cam Whitmore,13.5,0.2,0.4,0.40



Processing player_points_assists...
Saved 68 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,P.J. Washington,7.5,1.0,0.9,0.87
1,Evan Mobley,14.0,0.8,0.5,0.47
2,Kevin Durant,10.5,0.8,0.5,0.40
3,Yves Missi,6.5,0.8,0.6,0.53
4,Jalen Johnson,19.0,0.8,0.5,0.40
5,Keegan Murray,9.0,0.8,0.4,0.27
6,Ryan Rollins,9.5,0.6,0.7,0.67
7,Naz Reid,8.5,0.6,0.5,0.40
8,Ausar Thompson,8.0,0.6,0.5,0.53
9,Duncan Robinson,4.5,0.6,0.7,0.67



Processing player_turnovers...
Saved 5 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Darius Garland,2.5,0.8,0.5,0.33
1,Drew Eubanks,0.5,0.8,0.6,0.67
2,Josh Okogie,0.5,0.8,0.5,0.47
3,Jalen Johnson,3.5,0.6,0.7,0.60
4,Mouhamed Gueye,0.5,0.6,0.5,0.60



Processing player_blocks_steals...
Saved 14 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Micah Peavy,0.5,1.0,0.7,0.53
1,P.J. Washington,1.5,1.0,0.8,0.87
2,Trey Murphy III,1.5,0.6,0.7,0.67
3,Tyrese Martin,0.5,0.6,0.7,0.67
4,Nickeil Alexander-Walker,1.5,0.6,0.5,0.53
5,Mouhamed Gueye,1.5,0.6,0.5,0.47
6,Kawhi Leonard,1.5,0.6,0.6,0.60
7,Anthony Davis,2.5,0.6,0.6,0.40
8,Cooper Flagg,1.5,0.6,0.7,0.73
9,Josh Okogie,1.5,0.6,0.6,0.60


### underdog

In [14]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/underdog/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 64 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Saddiq Bey,26.5,1.0,0.7,0.60
1,Kawhi Leonard,31.5,1.0,0.7,0.60
2,Nickeil Alexander-Walker,28.5,1.0,0.8,0.60
3,Jalen Johnson,44.5,0.8,0.6,0.47
4,P.J. Washington,21.5,0.8,0.8,0.73
...,...,...,...,...,...
59,Jaden Ivey,13.5,0.2,0.2,0.13
60,Kyle Kuzma,20.5,0.2,0.4,0.40
61,Cade Cunningham,42.5,0.2,0.4,0.53
62,Jalen Duren,31.5,0.2,0.4,0.53



Processing player_points_rebounds...
Saved 35 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Nickeil Alexander-Walker,24.5,1.0,0.8,0.60
1,Saddiq Bey,23.5,1.0,0.7,0.60
2,Jeremiah Fears,20.5,0.8,0.6,0.60
3,P.J. Washington,19.5,0.8,0.8,0.80
4,Jalen Johnson,35.5,0.8,0.6,0.47
5,Norman Powell,28.5,0.8,0.7,0.60
6,Anthony Edwards,34.5,0.8,0.6,0.53
7,Kawhi Leonard,28.5,0.8,0.6,0.53
8,Trey Murphy III,26.5,0.6,0.6,0.60
9,Tobias Harris,18.5,0.6,0.5,0.47



Processing player_points_assists...
Saved 27 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Noah Clowney,6.5,0.6,0.4,0.40
1,Marvin Bagley III,8.5,0.6,0.3,0.27
2,Jaylon Tyson,7.5,0.6,0.7,0.53
3,Ryan Rollins,9.5,0.6,0.7,0.67
4,Rudy Gobert,11.5,0.6,0.7,0.60
5,Anthony Davis,13.5,0.6,0.6,0.40
6,Amen Thompson,14.5,0.6,0.5,0.40
7,Darius Garland,9.5,0.4,0.3,0.20
8,Anthony Edwards,9.5,0.4,0.4,0.33
9,Julius Randle,12.5,0.4,0.5,0.53



Processing player_turnovers...
Saved 4 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Darius Garland,2.5,0.8,0.5,0.33
1,Jalen Johnson,3.5,0.6,0.7,0.60
2,Julius Randle,2.5,0.6,0.5,0.47
3,Amen Thompson,2.5,0.6,0.5,0.53



Processing player_blocks_steals...
Saved 3 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Anthony Davis,2.5,0.6,0.6,0.40
1,Dyson Daniels,2.5,0.4,0.4,0.40
2,Evan Mobley,2.5,0.4,0.4,0.33
